# Datenbeschaffung: DAX 40 & MDAX 50 Preisdaten (2019–2025)

**CRISP-DM-Phase: Data Understanding**  
**Projekt:** Basket Trading using Bayesian Optimization  
**Autor:** Nassim Groppe

---

Dieses Notebook lädt historische Tages-Kursdaten und Fundamentaldaten für alle Bestandteile des DAX 40 und MDAX 50 herunter. Die Rohdaten werden in `data/raw/` gespeichert und dienen als Grundlage für alle nachfolgenden Analysen.

**Inhalt:**
1. Importe & Konfiguration
2. Ticker-Listen definieren
3. Preisdaten herunterladen & konsolidieren
4. Als Parquet speichern
5. Fundamentaldaten ziehen & als CSV speichern

## 1. Importe & Konfiguration

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from pathlib import Path

# Ausgabepfade
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Zeitraum
START_DATE = "2019-01-01"
END_DATE   = "2025-12-31"
INTERVAL   = "1d"

print(f"Zeitraum: {START_DATE} bis {END_DATE}, Intervall: {INTERVAL}")
print(f"Ausgabepfad: {RAW_DIR.resolve()}")

Zeitraum: 2019-01-01 bis 2025-12-31, Intervall: 1d
Ausgabepfad: /Users/lisagroppe/Desktop/DBU Data Analyst - Python /data/raw


## 2. Ticker-Listen definieren

Quelle: Wikipedia DAX / MDAX (Stand: Anfang 2025).  
Ticker-Format: Yahoo Finance `.DE`-Suffix (XETRA-Notierung).  
> **Hinweis:** Indexzusammensetzungen ändern sich halbjährlich (März/September). Bei Bedarf kann die Liste mit aktuellen STOXX-Daten abgeglichen werden.

In [2]:
dax40 = [
    "ADS.DE",  # Adidas
    "AIR.DE",  # Airbus
    "ALV.DE",  # Allianz
    "BAS.DE",  # BASF
    "BAYN.DE", # Bayer
    "BEI.DE",  # Beiersdorf
    "BMW.DE",  # BMW
    "BNR.DE",  # Brenntag
    "CBK.DE",  # Commerzbank
    "CON.DE",  # Continental
    "DTG.DE",  # Daimler Truck
    "DBK.DE",  # Deutsche Bank
    "DB1.DE",  # Deutsche Börse
    "DHL.DE",  # DHL Group
    "DTE.DE",  # Deutsche Telekom
    "EOAN.DE", # E.ON
    "FRE.DE",  # Fresenius
    "FME.DE",  # Fresenius Medical Care
    "G1A.DE",  # GEA Group
    "HNR1.DE", # Hannover Rück
    "HEI.DE",  # Heidelberg Materials
    "HEN3.DE", # Henkel Vz.
    "IFX.DE",  # Infineon
    "MBG.DE",  # Mercedes-Benz
    "MRK.DE",  # Merck KGaA
    "MTX.DE",  # MTU Aero Engines
    "MUV2.DE", # Munich Re
    "PAH3.DE", # Porsche SE
    "QIA.DE",  # Qiagen
    "RHM.DE",  # Rheinmetall
    "RWE.DE",  # RWE
    "SAP.DE",  # SAP
    "G24.DE",  # Scout24
    "SIE.DE",  # Siemens
    "ENR.DE",  # Siemens Energy
    "SHL.DE",  # Siemens Healthineers
    "SY1.DE",  # Symrise
    "VOW3.DE", # Volkswagen Vz.
    "VNA.DE",  # Vonovia
    "ZAL.DE",  # Zalando
]

mdax50 = [
    "AIXA.DE", # Aixtron
    "AT1.DE",  # Aroundtown
    "NDA.DE",  # Aurubis
    "BC8.DE",  # Bechtle
    "BFSA.DE", # Befesa
    "GBF.DE",  # Bilfinger
    "AFX.DE",  # Carl Zeiss Meditec
    "EVD.DE",  # CTS Eventim
    "DHER.DE", # Delivery Hero
    "LHA.DE",  # Deutsche Lufthansa
    "ECV.DE",  # Encavis
    "EVK.DE",  # Evonik
    "EVT.DE",  # Evotec
    "FRA.DE",  # Fraport
    "FNTN.DE", # Freenet
    "FPE3.DE", # Fuchs Petrolub Vz.
    "GXI.DE",  # Gerresheimer
    "HLE.DE",  # Hella
    "HFG.DE",  # HelloFresh
    "HAG.DE",  # Hensoldt
    "HOT.DE",  # Hochtief
    "BOSS.DE", # Hugo Boss
    "JEN.DE",  # Jenoptik
    "JUN3.DE", # Jungheinrich Vz.
    "SDF.DE",  # K+S
    "KIO.DE",  # Kion Group
    "KBX.DE",  # Knorr-Bremse
    "KRN.DE",  # Krones
    "LXS.DE",  # Lanxess
    "LEG.DE",  # LEG Immobilien
    "NEM.DE",  # Nemetschek
    "NDX1.DE", # Nordex
    "PUM.DE",  # Puma
    "RAA.DE",  # Rational
    "RDC.DE",  # Redcare Pharmacy
    "RRTL.DE", # RTL Group
    "WAF.DE",  # Siltronic
    "STM.DE",  # Stabilus
    "SAX.DE",  # Stroeer
    "TEG.DE",  # TAG Immobilien
    "TLX.DE",  # Talanx
    "TMV.DE",  # Traton
    "TKA.DE",  # ThyssenKrupp
    "8TRA.DE", # Traton (Stammaktie)
    "TUI1.DE", # TUI
    "UTDI.DE", # United Internet
    "WCH.DE",  # Wacker Chemie
    "MAN3.DE", # MAN SE
    "VH2.DE",  # Villeroy & Boch
    "CECONOMY.DE", # Ceconomy
]

all_tickers = dax40 + mdax50

print(f"DAX 40:  {len(dax40)} Ticker")
print(f"MDAX 50: {len(mdax50)} Ticker")
print(f"Gesamt:  {len(all_tickers)} Ticker")

DAX 40:  40 Ticker
MDAX 50: 50 Ticker
Gesamt:  90 Ticker


## 3. Preisdaten herunterladen & in Long-Format konsolidieren

Heruntergeladen werden: `Open`, `High`, `Low`, `Close`, `Volume`, `Adj Close`.  
Das Wide-Format (Ticker als Spalten) wird anschließend in ein Long-Format überführt, damit jede Zeile genau einem Ticker-Datum-Wert entspricht — ideal für nachfolgende Analysen und Datenbankoperationen.

In [3]:
print("Lade Preisdaten herunter...")

# Batch-Download: alle Ticker auf einmal für maximale Effizienz
raw_prices = yf.download(
    tickers=all_tickers,
    start=START_DATE,
    end=END_DATE,
    interval=INTERVAL,
    auto_adjust=True,   # Adj. Close direkt als Close
    progress=True,
)

print(f"\nShape (Wide-Format): {raw_prices.shape}")
print(f"Zeitraum: {raw_prices.index.min().date()} bis {raw_prices.index.max().date()}")
raw_prices.head(3)

Lade Preisdaten herunter...


[*********************100%***********************]  90 of 90 completed

4 Failed downloads:
['KIO.DE']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-01-01 -> 2025-12-31)')
['ECV.DE', 'CECONOMY.DE', 'MAN3.DE']: YFTzMissingError('possibly delisted; no timezone found')



Shape (Wide-Format): (1779, 454)
Zeitraum: 2019-01-02 bis 2025-12-30


Price        Adj Close                         Close                         \
Ticker     CECONOMY.DE ECV.DE KIO.DE MAN3.DE 8TRA.DE      ADS.DE     AFX.DE   
Date                                                                          
2019-01-02         NaN    NaN    NaN     NaN     NaN  170.651215  64.942871   
2019-01-03         NaN    NaN    NaN     NaN     NaN  170.234772  63.687710   
2019-01-04         NaN    NaN    NaN     NaN     NaN  176.805420  65.779640   

Price                                        ... Volume                 \
Ticker         AIR.DE   AIXA.DE      ALV.DE  ... TLX.DE TMV.DE TUI1.DE   
Date                                         ...                         
2019-01-02  76.132393  8.256731  118.047073  ...  99605    NaN  257269   
2019-01-03  73.418541  7.408050  116.805885  ...  86652    NaN  209047   
2019-01-04  77.027946  7.721120  119.639015  ...  85034    NaN  167200   

Price                                                                
Ticker     UTDI.DE VH2.DE   VNA.DE  VOW3.DE  WAF.DE  WCH.DE  ZAL.DE  
Date                                                                 
2019-01-02  308499    NaN  1618722  1116700  197681  123961  619245  
2019-01-03  299910    NaN  1396150   968713  359774  224729  831445  
2019-01-04  284338    NaN  1465083  1177680  298634  210853  757833  

[3 rows x 454 columns]

In [4]:
# Wide → Long-Format
# MultiIndex-Spalten (Preisart, Ticker) werden gestackt
prices_long = (
    raw_prices
    .stack(level=1)          # Ticker-Ebene nach unten
    .reset_index()
    .rename(columns={"level_1": "ticker", "Date": "date"})
)

# Index-Namen unterscheiden sich je nach yfinance-Version
prices_long.columns = [c.lower().replace(" ", "_") for c in prices_long.columns]

# Indexzugehörigkeit ergänzen
prices_long["index_name"] = prices_long["ticker"].apply(
    lambda t: "DAX40" if t in dax40 else "MDAX50"
)

# Zeilen ohne Kursdaten (nicht gehandelte Tage / delistete Titel) entfernen
prices_long.dropna(subset=["close"], inplace=True)
prices_long.sort_values(["ticker", "date"], inplace=True)
prices_long.reset_index(drop=True, inplace=True)

print(f"Shape (Long-Format): {prices_long.shape}")
print(f"Ticker mit Daten:    {prices_long['ticker'].nunique()}")
prices_long.head()

Shape (Long-Format): (150489, 9)
Ticker mit Daten:    86


,date,ticker,adj_close,close,high,low,open,volume,index_name
0,2019-07-01,8TRA.DE,NaN,20.949963,21.221632,20.788537,20.946026,1427241.0,MDAX50
1,2019-07-02,8TRA.DE,NaN,21.300375,21.402744,20.949962,21.036582,447352.0,MDAX50
2,2019-07-03,8TRA.DE,NaN,21.363371,21.536610,21.292502,21.402744,489627.0,MDAX50
3,2019-07-04,8TRA.DE,NaN,21.473614,21.473614,21.343685,21.410618,296932.0,MDAX50
4,2019-07-05,8TRA.DE,NaN,21.170446,21.465738,21.170446,21.465738,237111.0,MDAX50


In [5]:
# Kurze Datenqualitätsprüfung
print("=== Datenqualität ===")
print(f"Fehlende Werte:\n{prices_long.isnull().sum()}")
print(f"\nZeilenanzahl je Index:")
print(prices_long.groupby('index_name')['ticker'].count())
print(f"\nHandelstage gesamt: {prices_long['date'].nunique()}")

=== Datenqualität ===
Fehlende Werte:
date               0
ticker             0
adj_close     150489
close              0
high               0
low                0
open               0
volume             0
index_name         0
dtype: int64

Zeilenanzahl je Index:
index_name
DAX40     69967
MDAX50    80522
Name: ticker, dtype: int64

Handelstage gesamt: 1779


## 4. Preisdaten als Parquet speichern

Parquet bietet gegenüber CSV deutlich kleinere Dateien und erhält Datentypen (datetime, float) verlustfrei.

In [6]:
parquet_path = RAW_DIR / "dax_mdax_prices_2019_2025.parquet"
prices_long.to_parquet(parquet_path, index=False)

file_size_mb = parquet_path.stat().st_size / 1_048_576
print(f"Gespeichert: {parquet_path}")
print(f"Dateigröße:  {file_size_mb:.2f} MB")
print(f"Zeilen:      {len(prices_long):,}")

Gespeichert: ../data/raw/dax_mdax_prices_2019_2025.parquet
Dateigröße:  6.24 MB
Zeilen:      150,489


## 5. Fundamentaldaten herunterladen & als CSV speichern

Über `yfinance.Ticker.info` werden je Unternehmen folgende Felder gezogen:

| Feld | Bedeutung |
|---|---|
| `market_cap` | Marktkapitalisierung (EUR) |
| `pe_ratio` | Kurs-Gewinn-Verhältnis (trailing) |
| `dividend_yield` | Dividendenrendite |
| `sector` | Sektor (GICS) |
| `industry` | Branche |
| `full_name` | Vollständiger Unternehmensname |

> **Hinweis:** `Ticker.info` schickt für jeden Titel einen separaten HTTP-Request. Bei 90 Titeln dauert dieser Schritt mehrere Minuten.

In [7]:
import time

FUNDAMENTAL_FIELDS = {
    "longName":            "full_name",
    "marketCap":           "market_cap",
    "trailingPE":          "pe_ratio",
    "dividendYield":       "dividend_yield",
    "sector":              "sector",
    "industry":            "industry",
    "currency":            "currency",
    "country":             "country",
    "numberOfEmployees":   "employees",
    "priceToBook":         "price_to_book",
}

records = []
failed_tickers = []

for i, ticker in enumerate(all_tickers):
    try:
        info = yf.Ticker(ticker).info
        row = {"ticker": ticker}
        row["index_name"] = "DAX40" if ticker in dax40 else "MDAX50"
        for yf_key, col_name in FUNDAMENTAL_FIELDS.items():
            row[col_name] = info.get(yf_key, np.nan)
        records.append(row)
    except Exception as e:
        print(f"  Fehler bei {ticker}: {e}")
        failed_tickers.append(ticker)

    # Kurze Pause, um Rate-Limiting zu vermeiden
    time.sleep(0.3)

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(all_tickers)} Ticker verarbeitet...")

fundamentals_df = pd.DataFrame(records)
print(f"\nFertig. {len(fundamentals_df)} Einträge, {len(failed_tickers)} Fehler.")
if failed_tickers:
    print(f"Fehlgeschlagen: {failed_tickers}")
fundamentals_df.head()

  10/90 Ticker verarbeitet...
  20/90 Ticker verarbeitet...
  30/90 Ticker verarbeitet...
  40/90 Ticker verarbeitet...
  50/90 Ticker verarbeitet...
  60/90 Ticker verarbeitet...
  70/90 Ticker verarbeitet...
  80/90 Ticker verarbeitet...


404 Client Error: Not Found for url: https://query2.finance.yahoo.com/v10/finance/quoteSummary/CECONOMY.DE?modules=financialData%2CquoteType%2CdefaultKeyStatistics%2CassetProfile%2CsummaryDetail&corsDomain=finance.yahoo.com&formatted=false&symbol=CECONOMY.DE&crumb=XS9.5LeXsuh


  Fehler bei CECONOMY.DE: 'NoneType' object has no attribute 'update'
  90/90 Ticker verarbeitet...

Fertig. 89 Einträge, 1 Fehler.
Fehlgeschlagen: ['CECONOMY.DE']


,ticker,index_name,full_name,market_cap,pe_ratio,dividend_yield,sector,industry,currency,country,employees,price_to_book
0,ADS.DE,DAX40,adidas AG,2.522788e+10,18.426167,1.97,Consumer Cyclical,Footwear & Accessories,EUR,Germany,NaN,4.166545
1,AIR.DE,DAX40,Airbus SE,1.353088e+11,27.196203,1.85,Industrials,Aerospace & Defense,EUR,Netherlands,NaN,5.186012
2,ALV.DE,DAX40,Allianz SE,1.418303e+11,13.490054,4.64,Financial Services,Insurance - Diversified,EUR,Germany,NaN,2.261250
3,BAS.DE,DAX40,BASF SE,4.778894e+10,33.775000,4.23,Basic Materials,Chemicals,EUR,Germany,NaN,1.384576
4,BAYN.DE,DAX40,Bayer Aktiengesellschaft,3.767105e+10,NaN,NaN,Healthcare,Drug Manufacturers - General,EUR,Germany,NaN,1.451857


In [8]:
# Fundamentaldaten speichern
csv_path = RAW_DIR / "dax_mdax_fundamentals.csv"
fundamentals_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"Gespeichert: {csv_path}")
print(f"\nVerfügbarkeit der Fundamentaldaten:")
print(fundamentals_df.notnull().sum().to_string())

Gespeichert: ../data/raw/dax_mdax_fundamentals.csv

Verfügbarkeit der Fundamentaldaten:
ticker            89
index_name        89
full_name         87
market_cap        86
pe_ratio          75
dividend_yield    77
sector            86
industry          86
currency          86
country           86
employees          0
price_to_book     86


## Zusammenfassung

| Artefakt | Pfad | Beschreibung |
|---|---|---|
| Preisdaten (Parquet) | `data/raw/dax_mdax_prices_2019_2025.parquet` | Tages-OHLCV im Long-Format |
| Fundamentaldaten (CSV) | `data/raw/dax_mdax_fundamentals.csv` | Marktdaten je Ticker |

**Nächster Schritt:** `01_aufgabe9_regression_wine.ipynb` — oder direkt weiter mit der explorativen Datenanalyse der Kursdaten im Abschlussprojekt-Notebook.